### Building a chatbot with conversation history

This chatbot will be able to have a conversation and remember previous interactions.

In [12]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


load_dotenv()

True

##### Model without conversation history

In [7]:
model = init_chat_model(model="groq:llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000012491FCE960>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000012491FCDBB0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage

# input to model should be list of basemessage or prompt

user_message = [HumanMessage(content="Hi, Im Mounica. Currently learning langchain and langgraph")]

model.invoke(user_message)

AIMessage(content="Nice to meet you, Mounica. Langchain and Langgraph are exciting areas of research and development in natural language processing (NLP). Langchain refers to a type of AI that integrates multiple language models to create a more comprehensive and coherent understanding of language, while Langgraph is a framework for building graph-based language models.\n\nWhat are your goals in learning langchain and langgraph? Are you looking to apply these concepts in a specific project or industry, or are you simply interested in the theoretical aspects of NLP?\n\nAlso, what's your background in computer science and NLP? Have you worked with any other NLP libraries or frameworks, such as BERT, RoBERTa, or Transformers?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 49, 'total_tokens': 192, 'completion_time': 0.232088592, 'completion_tokens_details': None, 'prompt_time': 0.002393447, 'prompt_tokens_details': None, 'queue_time': 

In [10]:
user_message = [HumanMessage(content="Hi, Im Mounica. Currently learning langchain and langgraph"),
                AIMessage(content="Hello Mounica. Langchain and Langgraph are exciting topics in the field of natural language processing (NLP). Langchain focuses on creating a framework for building large language models and developing applications that utilize them, while Langgraph is a type of graph-based language model that aims to better represent complex linguistic structures.\n\nWhat specific aspects of Langchain and Langgraph are you interested in learning more about? Are you looking to build a project or implement these concepts in a particular application? I'd be happy to help you with any questions or provide guidance on getting started."),
                HumanMessage(content="do you remember my name?")]
model.invoke(user_message)

AIMessage(content="You're Mounica. I'll try to remember it for our conversation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 178, 'total_tokens': 195, 'completion_time': 0.035578157, 'completion_tokens_details': None, 'prompt_time': 0.010055469, 'prompt_tokens_details': None, 'queue_time': 0.0552181, 'total_time': 0.045633626}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7bd8-40a6-7091-8c4b-4725c6ebf335-0', usage_metadata={'input_tokens': 178, 'output_tokens': 17, 'total_tokens': 195})

In [11]:
model.invoke([HumanMessage(content="do you still remember my name?")])

AIMessage(content="I'm a large language model, I don't have personal memories or the ability to recall previous conversations or users' names. Each time you interact with me, it's a new conversation, and I don't retain any information from previous chats.\n\nSo, I don't remember your name, but I'm happy to chat with you and help with any questions or topics you'd like to discuss!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 42, 'total_tokens': 122, 'completion_time': 0.098876778, 'completion_tokens_details': None, 'prompt_time': 0.001931141, 'prompt_tokens_details': None, 'queue_time': 0.050241819, 'total_time': 0.100807919}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7bd9-389a-78a3-bcd5-bf8e90cddbab-0', usage_metadata={'input_tokens': 42, 'output_tokens': 80, 'total_tokens': 122})

##### Model with conversation history

We can use a Message History class to wrap our model and make it stateful.This will keep track of inputs and outputs of the model and store them in some datastore. Future interactions will then load these messages and pass them into the chain as part of the input.

In [13]:
session_store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    "this function gets the message history for the given session_id"
    if session_id not in session_store.keys():
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)    

In [15]:
config={"configurable":{"session_id":"user_1"}}

response = with_message_history.invoke(user_message, config=config)
response

AIMessage(content='Your name is Mounica.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 178, 'total_tokens': 186, 'completion_time': 0.009852717, 'completion_tokens_details': None, 'prompt_time': 0.011452128, 'prompt_tokens_details': None, 'queue_time': 0.054969152, 'total_time': 0.021304845}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7be5-5d83-76c1-a495-cfd5bd4b7af1-0', usage_metadata={'input_tokens': 178, 'output_tokens': 8, 'total_tokens': 186})

In [16]:
query="do you still remember me?"
response_b = with_message_history.invoke(query, config=config)
response_b


AIMessage(content='Yes, I still remember your name as Mounica. It was nice chatting with you earlier about Langchain and Langgraph. How can I assist you further today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 201, 'total_tokens': 236, 'completion_time': 0.048613893, 'completion_tokens_details': None, 'prompt_time': 0.013000182, 'prompt_tokens_details': None, 'queue_time': 0.055207858, 'total_time': 0.061614075}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7be6-35fd-72b0-b3fa-e1a51a0637dc-0', usage_metadata={'input_tokens': 201, 'output_tokens': 35, 'total_tokens': 236})

#### Prompt Templates

Prompt templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated.
First, let's add in a system message with some custom instructions (but still making messagges as input). Next we'll add in more input besides just the messages.

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model
chain


ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [19]:
chain.invoke({"messages":[HumanMessage(content="give few lines about agentic ai")]})

AIMessage(content='**Agentive AI: A New Frontier**\n\nAgentive AI refers to artificial intelligence (AI) that can take initiative, make decisions, and act autonomously in complex situations. This type of AI is designed to be proactive, rather than just reactive, and can adapt to changing environments and goals.\n\nKey characteristics of agentive AI include:\n\n1. **Autonomy**: Agentive AI systems can make decisions and take actions without human intervention.\n2. **Proactivity**: Agentive AI can anticipate and respond to emerging situations, rather than just reacting to inputs.\n3. **Goal-oriented behavior**: Agentive AI systems are designed to achieve specific goals and objectives.\n4. **Self-modifying behavior**: Agentive AI can modify its own behavior and decision-making processes based on new information or experiences.\n\nExamples of agentive AI include:\n\n1. **Self-driving cars**: Autonomous vehicles that can navigate complex roads and respond to unexpected events.\n2. **Persona

In [20]:
chain_with_message_history = RunnableWithMessageHistory(runnable=chain,
                                                        get_session_history=get_session_history)

config = {"configurable":{"session_id":"chain_user"}}


In [21]:
user_input={"messages":[
    HumanMessage(content="get me few lines about langchain")
]}

result= chain_with_message_history.invoke(user_input, config=config)
result

AIMessage(content="LangChain is an open-source library for building chain-based LLM (Large Language Model) workflows. It's designed to simplify the creation of complex AI applications by allowing developers to build and manage chains of LLM operations. These chains, or workflows, can be composed of multiple models, data sources, and tasks to achieve more sophisticated AI tasks, such as dialogue systems, question answering, and more.\n\nLangChain provides a flexible and modular architecture, making it easy to integrate with various LLMs and data sources, such as Hugging Face Transformers, LLaMA, and more. It also offers tools for model and data management, as well as debugging and logging capabilities.\n\nThe library is maintained by a community of developers and is widely used in various AI applications, including chatbots, virtual assistants, and more.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 58, 'total_tokens': 222, 'comple

In [22]:
user_input={"messages":[
    HumanMessage(content="how are tools used in it")
]}
result_b = chain_with_message_history.invoke(user_input, config=config)

result_b

AIMessage(content='LangChain provides several tools that can be used to build and manage chain-based LLM workflows. Here are some of the key tools:\n\n1. **LLM Chain**: This is the core component of LangChain, allowing developers to create and manage chains of LLM operations. Chains can be composed of multiple models, data sources, and tasks.\n\n2. **Model Manager**: This tool allows developers to manage LLM models, including loading, saving, and versioning models. It also provides tools for model fine-tuning and experimentation.\n\n3. **Dataset Manager**: This tool provides a way to manage datasets, including loading, saving, and processing datasets. It also provides tools for data cleaning, filtering, and transformation.\n\n4. **Task Manager**: This tool allows developers to define and manage tasks, including task workflows, task outputs, and task inputs.\n\n5. **Debugger**: This tool provides a way to debug LLM chains, including tools for inspecting chain outputs, setting breakpoint

##### With multiple input variables

In [23]:
history_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant, answer the asked question in this language{language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
history_prompt

ChatPromptTemplate(input_variables=['language', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langch

In [24]:
history_chain = history_prompt | model
history_chain

ChatPromptTemplate(input_variables=['language', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langch

In [25]:
runnable_history_chain = RunnableWithMessageHistory(
    runnable=history_chain,
    get_session_history=get_session_history,
    input_messages_key="messages"
)


In [26]:
config = {"configurable":{"session_id":"history_user"}}

response = runnable_history_chain.invoke(
    {"messages":[HumanMessage(content="Hi, My name is Mounica")], "language":"hindi"},
    config=config
)

response

AIMessage(content='नमस्ते मौणिका, मुझे खुशी हुई आपका परिचय प्राप्त करने का। मैं आपकी सहायता के लिए यहाँ हूँ। क्या आपके पास कोई विशिष्ट प्रश्न या विषय है जिस पर चर्चा करना चाहते हैं?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 58, 'total_tokens': 142, 'completion_time': 0.11964157, 'completion_tokens_details': None, 'prompt_time': 0.003907421, 'prompt_tokens_details': None, 'queue_time': 0.054834768, 'total_time': 0.123548991}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7c05-fde6-7e00-89d0-b1329bc17c2a-0', usage_metadata={'input_tokens': 58, 'output_tokens': 84, 'total_tokens': 142})

In [27]:
response = runnable_history_chain.invoke(
    {"messages":[HumanMessage(content="what is my name")], "language":"telugu"},
    config=config
)

response

AIMessage(content='మీ పేరు మౌనిక.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 156, 'total_tokens': 180, 'completion_time': 0.031836817, 'completion_tokens_details': None, 'prompt_time': 0.009429973, 'prompt_tokens_details': None, 'queue_time': 0.061795147, 'total_time': 0.04126679}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7c07-7572-7923-8573-872614fd24f5-0', usage_metadata={'input_tokens': 156, 'output_tokens': 24, 'total_tokens': 180})

#### Managing the conversation history

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

"trim_messages" helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.


In [29]:
from langchain_core.messages.utils import trim_messages
from langchain_core.messages import SystemMessage

trimmer = trim_messages(
    max_tokens=50,
    token_counter=model,
    strategy="last",
    allow_partial=True,
    start_on="human"
)

trimmer


RunnableLambda(...)

In [33]:
user_messages=[
    SystemMessage(content="you are helpful assistant, answer user questions"),
    HumanMessage(content="Hi My name is Mounica"),
    AIMessage(content="Hi Mounica, please let me know how can I help you?"),
    HumanMessage(content="what is 2 + 2?"),
    AIMessage(content="it is 4"),
    HumanMessage(content="I like icecreams"),
    AIMessage(content="Wow, good taste")

]

trimmed_message=trimmer.invoke(user_messages)

trimmed_message

[HumanMessage(content='what is 2 + 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='it is 4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like icecreams', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Wow, good taste', additional_kwargs={}, response_metadata={})]

##### Applying it in chain

In [34]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer) | prompt |model
chain

RunnableAssign(mapper={
  messages: RunnableLambda(itemgetter('messages'))
            | RunnableLambda(...)
})
| ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[lang

In [39]:
chain.invoke({"messages":user_messages + [HumanMessage(content="do I like icrecreams?")]})

AIMessage(content='You told me earlier that you like icecreams.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 100, 'total_tokens': 112, 'completion_time': 0.018667914, 'completion_tokens_details': None, 'prompt_time': 0.005790313, 'prompt_tokens_details': None, 'queue_time': 0.050210426, 'total_time': 0.024458227}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7c2d-7ef5-7550-bfdc-7abc6d3f3581-0', usage_metadata={'input_tokens': 100, 'output_tokens': 12, 'total_tokens': 112})

#### With language input

In [40]:
chain_lang = RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)  | history_prompt | model
chain_lang

RunnableAssign(mapper={
  messages: RunnableLambda(itemgetter('messages'))
            | RunnableLambda(...)
})
| ChatPromptTemplate(input_variables=['language', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.An

In [42]:
chain_lang.invoke({"messages":user_messages + [HumanMessage(content="do I like icrecreams?")],
                   "language":"telugu"})

AIMessage(content='లేదు, మీరు ఆయిస్ క్రీం ను పసిజె చేశారు.. వాస్తవానికి మీరు ఆయిస్ క్రీం ను ఇష్టపడతారని పేర్కొన్నారు.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 100, 'total_tokens': 266, 'completion_time': 0.249775956, 'completion_tokens_details': None, 'prompt_time': 0.005495166, 'prompt_tokens_details': None, 'queue_time': 0.054347623, 'total_time': 0.255271122}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7c33-480a-7a31-ae76-378016409ff4-0', usage_metadata={'input_tokens': 100, 'output_tokens': 166, 'total_tokens': 266})